# Setup Chroma Vector Search
Builds a persistent ChromaDB collection from `data/walmart_products.db` using OpenAI `text-embedding-3-small`.

In [1]:
from pathlib import Path
import json
import sqlite3
from dotenv import load_dotenv
from chromadb import PersistentClient
from chromadb.utils import embedding_functions


In [2]:
workspace = Path.cwd().resolve().parents[0] if (Path.cwd() / 'data').exists() is False else Path.cwd()
if not (workspace / 'data').exists():
    workspace = Path.cwd().resolve().parents[1]
backend_env = workspace / 'backend' / '.env'
load_dotenv(backend_env)
db_path = workspace / 'data' / 'walmart_products.db'
persist_dir = (workspace / 'data' / 'chroma_store').resolve()
persist_dir.mkdir(parents=True, exist_ok=True)
db_path, persist_dir

(WindowsPath('C:/Users/Administrator/SB-OAI-FDE-Training/Week6/day4/data/walmart_products.db'),
 WindowsPath('C:/Users/Administrator/SB-OAI-FDE-Training/Week6/day4/data/chroma_store'))

In [3]:
backend_env

WindowsPath('C:/Users/Administrator/SB-OAI-FDE-Training/Week6/day4/backend/.env')

In [3]:
import os
api_key = os.getenv('OPENAI_API_KEY', '').strip().strip('\"')
if not api_key:
    raise ValueError('OPENAI_API_KEY is missing in backend/.env')
embed_model = os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small')
chat_model = os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini')
embed_model, chat_model

('text-embedding-3-small', 'gpt-4o-mini')

In [4]:
conn = sqlite3.connect(db_path)
conn.row_factory = sqlite3.Row
rows = conn.execute('SELECT * FROM products').fetchall()
conn.close()
len(rows)


1000

In [5]:
def safe_json(raw, default):
    try:
        return json.loads(raw) if raw else default
    except Exception:
        return default

def doc_text(r):
    specs = safe_json(r['specifications'], [])
    reviews = safe_json(r['customer_reviews'], [])
    colors = safe_json(r['colors'], [])
    review_text = ' '.join(str(x.get('review', '')) for x in reviews[:5])
    spec_text = ' '.join(f"{x.get('name', '')} {x.get('value', '')}" for x in specs)
    return ' '.join([
        str(r['product_name'] or ''),
        str(r['description'] or ''),
        str(r['category_name'] or ''),
        str(r['root_category_name'] or ''),
        str(r['ingredients'] or ''),
        str(r['brand'] or ''),
        spec_text,
        ' '.join(colors),
        review_text
    ])

def est_tokens(text: str) -> int:
    return max(1, len(text) // 4)

def iter_batches(rows, max_est_tokens=120000, max_items=64):
    ids, docs, metas = [], [], []
    token_total = 0
    for r in rows:
        pid = str(r['product_id'] or '')
        if not pid:
            continue
        text = doc_text(r)
        est = est_tokens(text)
        meta = {
            'product_id': pid,
            'product_name': str(r['product_name'] or ''),
            'category_name': str(r['category_name'] or ''),
            'root_category_name': str(r['root_category_name'] or ''),
            'rating': float(r['rating'] or 0),
            'final_price': float(r['final_price'] or 0)
        }

        if ids and (token_total + est > max_est_tokens or len(ids) >= max_items):
            yield ids, docs, metas
            ids, docs, metas = [], [], []
            token_total = 0

        ids.append(pid)
        docs.append(text)
        metas.append(meta)
        token_total += est

    if ids:
        yield ids, docs, metas



In [6]:
client = PersistentClient(path=str(persist_dir))
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=api_key,
    model_name=embed_model
)
collection = client.get_or_create_collection(
    name='walmart_products',
    embedding_function=embedding_fn,
    metadata={'hnsw:space': 'cosine'}
)

batch_count = 0
total_upserted = 0
for ids, documents, metadatas in iter_batches(rows):
    collection.upsert(ids=ids, documents=documents, metadatas=metadatas)
    batch_count += 1
    total_upserted += len(ids)
    print(f'Batch {batch_count}: upserted {len(ids)} docs (total={total_upserted})')

print(f'Completed upsert: {total_upserted} products into {persist_dir}')



Batch 1: upserted 64 docs (total=64)
Batch 2: upserted 64 docs (total=128)
Batch 3: upserted 64 docs (total=192)
Batch 4: upserted 64 docs (total=256)
Batch 5: upserted 64 docs (total=320)
Batch 6: upserted 64 docs (total=384)
Batch 7: upserted 64 docs (total=448)
Batch 8: upserted 64 docs (total=512)
Batch 9: upserted 64 docs (total=576)
Batch 10: upserted 64 docs (total=640)
Batch 11: upserted 64 docs (total=704)
Batch 12: upserted 64 docs (total=768)
Batch 13: upserted 64 docs (total=832)
Batch 14: upserted 64 docs (total=896)
Batch 15: upserted 64 docs (total=960)
Batch 16: upserted 40 docs (total=1000)
Completed upsert: 1000 products into C:\Users\Administrator\SB-OAI-FDE-Training\Week6\day4\data\chroma_store


In [8]:
result = collection.query(query_texts=['affordable organic skincare with good reviews'], n_results=5)
list(zip(result['ids'][0], result['distances'][0]))

[('910055282', 0.5405073761940002),
 ('2010024322', 0.5656015872955322),
 ('3440558193', 0.5668262243270874),
 ('45696491', 0.5699852705001831),
 ('5201103967', 0.5911872982978821)]